# S6.P1 · 配对比较判据的纯合成校准

**问题**：DR-MX-002 把项目的显著性判据从「跨 ≥5 seed 且两模型区间不重叠」改为
「配对差的 95% 区间不含 0（训练 seed × 测试客户簇两层重抽样）并给 p 值」，
并要求每个比较声明测试集正例数与可检出差异（S0.1P 近似）。它们各自留了推翻条件：

- **D-1**：零效应下新判据的第一类错误率若 > 0.07，回退旧判据；
- **D-2**：以 S0.1P 预测的可检出差异为真实差距时，功效若不在 [0.707, 0.879]（即所需正例数偏差 > 25%），改用模拟法。

本 notebook 在合成设定下检验这两条，并顺带比较旧判据与 seed 级符号翻转检验。
逻辑在 `modules/m6_evaluation/components/paired_eval.py`，参数在 `modules/m6_evaluation/configs/s6_p1_calibration.yaml`。

> 全部为程序生成的合成打分：不使用任何代理或真实数据，不训练模型，不运行隐私集合求交（PSI）或攻击。

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
PCT = 100          # 比例换算为百分数（仅显示）
ROUND_RATE = 3     # 拒绝率、相关系数的显示精度
ROUND_GAP = 4      # AUROC 差与标准误的显示精度

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m6_evaluation/configs/s6_p1_calibration.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True,
                                  cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", cfg["step_id"], "|", cfg["purpose"])
rng = np.random.default_rng(cfg["seed"])
RESULTS = REPO_ROOT / "modules/m6_evaluation/results"

config: modules/m6_evaluation/configs/s6_p1_calibration.yaml sha256:ba57a9fa5a31
seed: 20261025
git: e36a47c（工作区含本步未提交改动）
step: S6.P1 | 在零效应与已知效应的合成设定下，检验新判据的第一类错误率、与旧判据和符号翻转检验的功效对比，以及 S0.1P 可检出差异近似的准确度


In [2]:
from modules.m6_evaluation.components import paired_eval as PE

auc_a = PE.population_auc(cfg["base_rate"], cfg["signal"], cfg["sigma_a"], cfg["n_large"],
                          np.random.default_rng(cfg["seed"]))
print("模型 A 的总体 AUROC ≈ %.3f（正例率 %.0f%%）" % (auc_a, cfg["base_rate"] * PCT))
sys_a = PE.systematic_sigma(cfg["sigma_a"], cfg["sigma_seed"])
print("模型 A：系统误差 %.3f，seed 误差 %.3f" % (sys_a, cfg["sigma_seed"]))
flip = pd.DataFrame({"seed 数": cfg["sign_flip_seed_range"],
                     "符号翻转最小双侧 p": [PE.min_sign_flip_p(k) for k in cfg["sign_flip_seed_range"]]})
print(flip.to_string(index=False))

模型 A 的总体 AUROC ≈ 0.845（正例率 10%）
模型 A：系统误差 0.490，seed 误差 0.100
 seed 数  符号翻转最小双侧 p
      5    0.062500
      6    0.031250
      7    0.015625
      8    0.007812


**读法**：5 个 seed 时符号翻转检验能达到的最小 p 值是 0.0625，按 0.05 判断永远不能通过；6 个 seed 才降到 0.031。

In [3]:
null_rows = []
for n_pos in cfg["n_pos_targets"]:
    out = PE.calibration_run(n_pos, cfg["base_rate"], cfg["signal"], sys_a, sys_a,
                             cfg["sigma_seed"], cfg["n_seeds"], cfg["n_rep_null"], cfg["n_boot"],
                             cfg["alpha"], cfg["power"], rng)
    null_rows.append(out)
null = pd.DataFrame(null_rows)
null.to_csv(RESULTS / "s6p1_type1_error.csv", index=False)
view = null[["n_pos_target", "mean_positives", "reject_rate_new_rule", "reject_rate_old_rule_5seed",
             "reject_rate_sign_flip_full", "reject_rate_sign_flip_5seed", "mean_estimate_correlation"]]
view.columns = ["目标正例数", "实际正例均值", "新判据", "旧判据(5 seed)", "符号翻转(6 seed)",
                "符号翻转(5 seed)", "AUROC 估计相关"]
print("零效应（两模型总体 AUROC 相同）下的拒绝率，每档 %d 次重复：" % cfg["n_rep_null"])
print(view.round(ROUND_RATE).to_string(index=False))
d1_pass = bool((null["reject_rate_new_rule"] <= cfg["type1_ceiling"]).all())
print("D-1 推翻条件（第一类错误率 > %.2f）是否触发：%s" % (cfg["type1_ceiling"], "否" if d1_pass else "是"))

零效应（两模型总体 AUROC 相同）下的拒绝率，每档 400 次重复：
 目标正例数  实际正例均值   新判据  旧判据(5 seed)  符号翻转(6 seed)  符号翻转(5 seed)  AUROC 估计相关
   150 149.532 0.038          0.0          0.81           0.0       0.687
   441 440.242 0.045          0.0          0.80           0.0       0.687
D-1 推翻条件（第一类错误率 > 0.07）是否触发：否


**读法**：

- **新判据**的拒绝率应接近名义的 0.05；高于 0.07 即触发 D-1 推翻条件。
- **旧判据**在零效应下几乎从不拒绝——它过于保守，代价在下一格的功效对比里。
- **seed 级符号翻转检验**只反映训练随机性：它固定了测试集，把两个模型在这一份测试集上的系统差异当成真实差异，
  所以在「两模型总体上一样好」的零效应下拒绝率远高于 0.05。它回答的是「在这一份测试集上」的条件问题，
  不能用来下总体结论（仓库 README 已称其为「条件符号翻转」）。
- **AUROC 估计相关**：两个系统误差相互独立的模型约为 0.69，低于 S0.1P 默认的 0.8；
  相关越低所需正例越多（约按 (1 − r) 成比例），所以正式比较必须代入重抽样估计的相关，而不是默认值。
- **与 D-1 原文的出入**：D-1 写的是「在 M2 零效应合成设定下重复 ≥200 次」。本格用的是打分层面的合成设定
  （直接生成两个模型的打分，不训练模型），检验的是判据本身；在 M2 生成器上训练模型再比较属于实验，
  暂停期间不做，留待恢复后补一次。

In [4]:
from modules.m0_compliance.components.validation_power import paired_auc_mde

alt_rows = []
for n_pos, row in zip(cfg["n_pos_targets"], null_rows):
    corr = row["mean_estimate_correlation"]
    gap = paired_auc_mde(auc_a, n_pos, cfg["base_rate"], corr, cfg["alpha"], cfg["power"])
    total_b = PE.sigma_for_gap(cfg["base_rate"], cfg["signal"], cfg["sigma_a"], gap, cfg["n_large"],
                               cfg["seed"], cfg["sigma_search_max"])
    sys_b = PE.systematic_sigma(total_b, cfg["sigma_seed"])
    out = PE.calibration_run(n_pos, cfg["base_rate"], cfg["signal"], sys_a, sys_b,
                             cfg["sigma_seed"], cfg["n_seeds"], cfg["n_rep_alt"], cfg["n_boot"],
                             cfg["alpha"], cfg["power"], rng)
    out.update({"true_gap": gap, "sigma_total_b": total_b, "estimate_correlation_used": corr})
    alt_rows.append(out)
alt = pd.DataFrame(alt_rows)
alt.to_csv(RESULTS / "s6p1_power_at_predicted_mde.csv", index=False)
view = alt[["n_pos_target", "estimate_correlation_used", "true_gap", "mean_diff", "reject_rate_new_rule",
            "reject_rate_old_rule_5seed", "reject_rate_sign_flip_5seed"]]
view.columns = ["目标正例数", "代入的估计相关", "设定的总体差距(S0.1P 预测)", "观察差均值", "新判据功效",
                "旧判据功效(5 seed)", "符号翻转功效(5 seed)"]
print("真实差距 = S0.1P 预测的可检出差异时，每档 %d 次重复：" % cfg["n_rep_alt"])
print(view.round(ROUND_GAP).to_string(index=False))
lo, hi = cfg["power_band"]
d2_pass = bool(alt["reject_rate_new_rule"].between(lo, hi).all())
print("D-2 推翻条件（功效不在 [%.3f, %.3f]）是否触发：%s" % (lo, hi, "否" if d2_pass else "是"))

真实差距 = S0.1P 预测的可检出差异时，每档 300 次重复：
 目标正例数  代入的估计相关  设定的总体差距(S0.1P 预测)  观察差均值  新判据功效  旧判据功效(5 seed)  符号翻转功效(5 seed)
   150   0.6867             0.0451 0.0449 0.7733         0.1233             0.0
   441   0.6874             0.0263 0.0266 0.8800         0.0967             0.0
D-2 推翻条件（功效不在 [0.707, 0.879]）是否触发：是


**读法**：

- 正例 150 时功效约 0.77，在区间内；正例 441 时功效 0.880，比区间上限 0.879 高 0.001，
  **按预先写定的阈值，D-2 推翻条件触发**。300 次重复的蒙特卡洛标准误约 0.019，这个超出量在模拟误差以内，
  但阈值是事先定的，不因「只差一点」而改判。
- 方向：功效高于 0.8，说明 S0.1P 近似在这个设定下**高估**了所需正例（约 25%），偏保守，不会让项目误以为测试集够大。
- 备择下模型 B 噪声更大，两模型估计相关降到 0.55–0.60，低于零效应下的 0.69；代入的是零效应下的相关，
  近似仍偏保守。
- 旧判据的功效只有 0.10–0.12；5 个 seed 的符号翻转功效恒为 0。
- D-2 的预定后果是「改用模拟法，并重算 offline_auc_positives.csv」，在下一格执行。
- 上表「新判据功效」所用的「S0.1P 预测差距」是代入实测相关 0.69 得到的；S0.1P 默认的 0.8 会给出更小的预测差距。

In [5]:
from modules.m0_compliance.components.validation_power import positives_for_paired_auc_mde

se_rows = []
for label, frame in (("零效应", null), ("备择", alt)):
    for _, r in frame.iterrows():
        se_rows.append({"设定": label, "目标正例数": int(r["n_pos_target"]),
                        "重抽样标准误均值": r["mean_bootstrap_se"],
                        "跨重复的真实标准差": r["sd_diff_across_reps"],
                        "比值": r["mean_bootstrap_se"] / r["sd_diff_across_reps"]})
se_view = pd.DataFrame(se_rows)
print("配对差的重抽样标准误 vs 跨重复的真实标准差：")
print(se_view.round(ROUND_GAP).to_string(index=False))
lo, hi = cfg["bootstrap_se_ratio_band"]
se_ok = bool(se_view["比值"].between(lo, hi).all())
print("重抽样标准误比值是否全部落在 [%.3f, %.3f]：%s" % (lo, hi, "是" if se_ok else "否"))

# D-2 的预定后果：用模拟法重算所需正例数。标准差在备择设定下估计（模型 B 的总体 AUROC 比 A 低目标差距）
pos_rows = []
for mde in cfg["mde_targets"]:
    sim = PE.simulated_positives(mde, cfg["sd_start_positives"], cfg["base_rate"], cfg["signal"],
                                 cfg["sigma_a"], cfg["sigma_seed"], cfg["n_seeds"], cfg["n_rep_sd"],
                                 cfg["alpha"], cfg["power"], cfg["n_large"], cfg["seed"],
                                 cfg["sigma_search_max"], rng)
    r_null = float(null["mean_estimate_correlation"].mean())
    pos_rows.append({
        "目标差距": mde,
        "模拟法所需正例": sim["required_positives"],
        "S0.1P 近似（相关 %.2f）" % r_null: positives_for_paired_auc_mde(
            auc_a, mde, cfg["base_rate"], r_null, cfg["alpha"], cfg["power"]),
        "S0.1P 近似（默认相关 %.1f）" % cfg["s01p_default_correlation"]: positives_for_paired_auc_mde(
            auc_a, mde, cfg["base_rate"], cfg["s01p_default_correlation"], cfg["alpha"], cfg["power"]),
    })
pos = pd.DataFrame(pos_rows)
pos.to_csv(RESULTS / "s6p1_positives_simulated.csv", index=False)
print()
print("所需正例数：模拟法（备择设定）vs S0.1P 近似，模型 A 总体 AUROC ≈ %.3f：" % auc_a)
print(pos.to_string(index=False))

配对差的重抽样标准误 vs 跨重复的真实标准差：
 设定  目标正例数  重抽样标准误均值  跨重复的真实标准差     比值
零效应    150    0.0124     0.0120 1.0335
零效应    441    0.0072     0.0068 1.0618
 备择    150    0.0162     0.0182 0.8905
 备择    441    0.0085     0.0083 1.0250
重抽样标准误比值是否全部落在 [0.894, 1.118]：否



所需正例数：模拟法（备择设定）vs S0.1P 近似，模型 A 总体 AUROC ≈ 0.845：
 目标差距  模拟法所需正例  S0.1P 近似（相关 0.69）  S0.1P 近似（默认相关 0.8）
 0.01     1773               3045                1947
 0.02      693                762                 487
 0.03      293                339                 217


**读法**：

- **重抽样标准误**：四个格中三个落在 [0.894, 1.118]；备择设定、150 个正例时比值约 0.89，略低于下限。
  即测试集小到 150 个正例时，重抽样标准误会把备择下的真实标准差低估约一成，单用它算可检出差异会偏乐观。
  零效应下两档都在区间内，第一类错误率不受影响（与 D-1 一致）。
- 因此正式报告的可检出差异取**两种算法中较大者**：`max(bootstrap, closed_form)`（evaluation_protocol v1 第四节）。
  正例较多时闭式近似偏保守、较少时重抽样偏乐观，取大者在本校准覆盖的两档都不乐观。
- **所需正例数（D-2 的预定后果，模拟法重算）**：备择设定下，检出 0.02 约需 693 个正例、0.03 约需 293 个、0.01 约需 1,773 个。
  所需正例与标准差的平方成正比；300 次重复的标准差估计相对误差约 4%，所需正例的 95% 误差带约 ±16%。
- S0.1P 近似代入实测相关约 0.69 时**偏保守**：0.02、0.03 两档多估约 10%–16%，0.01 档多估约七成（该档只作量级参照）。
- **代入默认相关 0.8 时，0.02、0.03 两档少估约三成**（0.02 只要 487 个），即 S0.1P 已发布的「441 个正例检出 0.02」偏乐观；0.01 档反而多估约一成。
- 新结果另存为 `s6p1_positives_simulated.csv`；S0.1P 的 `offline_auc_positives.csv` 不改，其规划用途由本文件替代。
- 这些数都依赖合成设定（潜变量 + 两层误差），对真实测试集只是量级参照；真实比较以那一次比较自己的重抽样结果为准。

## 结论

结论按上方两格的实际输出写入 `modules/m6_evaluation/evaluation_protocol.md` 第七节，
并决定 DR-MX-002 · D-1、D-2 是否保留：

- **D-1 未触发**：新判据在零效应下的第一类错误率为 0.04 左右，在名义 0.05 附近，保留。M2 生成器上的复核留待恢复后；
- **D-2 触发**（正例 441 时功效 0.880，超出上限 0.001）：按预定后果用模拟法重算所需正例，见 `s6p1_positives_simulated.csv`
  ——检出 0.02 约需 693 个正例，而不是 S0.1P 发布的 441 个；S0.1P 闭式近似只作粗估，且须代入约 0.7 而不是 0.8 的相关；
- 正式报告的可检出差异取重抽样法与闭式近似中**较大者**：150 个正例时重抽样标准误在备择下偏低约一成；
- 旧判据的功效约 0.1，5 个 seed 的符号翻转检验功效恒为 0；6 个 seed 的符号翻转在零效应下拒绝率约 0.8，不能用于总体结论。

> 合成设定只检验判据本身的统计性质，不对任何模型或业务效果下结论。